# Local Document Reader
## Notebook 02: Embeddings and FAISS Vector Database
\n
### Objective
This notebook turns the processed chunks from Phase 2 into searchable numerical vectors. It uses a local Ollama embedding model, normalizes its vectors, stores them in a FAISS index, and retrieves the most relevant source passages for a natural-language question.


### 1. One-time dependency and model check\n
Run this command in the project terminal before executing the embedding cells: `venv/bin/python -m pip install faiss-cpu ollama`. Then ensure Ollama is running and the `nomic-embed-text` model is present with `ollama list`; if needed run `ollama pull nomic-embed-text`.\n
\n
Why: FAISS handles vector search and Ollama generates vectors locally, so no cloud API key is required.

In [6]:
import json\n
import sys\n
from pathlib import Path\n
from typing import Any\n
\n
import faiss\n
import numpy as np\n
import ollama\n
import pandas as pd\n
\n
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()\n
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'\n
VECTOR_DB_DIR = PROJECT_ROOT / 'data' / 'vector_store'\n
VECTOR_DB_DIR.mkdir(parents=True, exist_ok=True)\n
EMBEDDING_MODEL = 'nomic-embed-text'\n
\n
print('Libraries imported successfully!')\n
print('Processed chunks:', PROCESSED_DIR)\n
print('Vector store:', VECTOR_DB_DIR)\n
print('Embedding model:', EMBEDDING_MODEL)

SyntaxError: unexpected character after line continuation character (1185228010.py, line 1)

### 2. Load processed chunks\n
An embedding only represents the chunk text. The remaining fields — especially source and page number — remain in a parallel metadata list, so a FAISS result can be turned back into a citable document passage.

In [ ]:
chunk_files = sorted(PROCESSED_DIR.glob('*_chunks.json'))\n
if not chunk_files:\n
    raise FileNotFoundError('No processed chunk JSON files found. Run Notebook 01 first.')\n
\n
all_chunks: list[dict[str, Any]] = []\n
for file_path in chunk_files:\n
    with file_path.open(encoding='utf-8') as file:\n
        all_chunks.extend(json.load(file))\n
\n
if not all_chunks:\n
    raise ValueError('Processed chunk files were empty.')\n
\n
chunks_df = pd.DataFrame(all_chunks)\n
print('Processed files found:', len(chunk_files))\n
print('Total chunks loaded:', len(all_chunks))\n
display(chunks_df[['chunk_id', 'source', 'page_number', 'character_count', 'text']].head())

### 3. Generate embeddings with task prefixes\n
An embedding is a numerical vector that represents the meaning of text. Nomic uses `search_document:` for source chunks and `search_query:` for questions. Using the correct prefix keeps document and question embeddings compatible in the same search space.\n
\n
Expected result: an array with one vector per input text. The model determines the vector dimension; this notebook does not hard-code 768.

In [ ]:
def _embedding_values(response: Any) -> list[list[float]]:\n
    """Support both dict-like and current Ollama client responses."""\n
    return response['embeddings'] if isinstance(response, dict) else response.embeddings\n
\n
def generate_embeddings(texts: list[str], model: str = EMBEDDING_MODEL, batch_size: int = 32) -> np.ndarray:\n
    """Embed document text in batches, preserving input order."""\n
    if not texts:\n
        raise ValueError('Input text list is empty.')\n
    if batch_size <= 0:\n
        raise ValueError('Batch size must be positive.')\n
\n
    vectors: list[list[float]] = []\n
    for start in range(0, len(texts), batch_size):\n
        batch = ['search_document: ' + text.strip() for text in texts[start:start + batch_size]]\n
        response = ollama.embed(model=model, input=batch)\n
        vectors.extend(_embedding_values(response))\n
\n
    embeddings = np.ascontiguousarray(vectors, dtype=np.float32)\n
    if embeddings.ndim != 2 or embeddings.shape[0] != len(texts):\n
        raise ValueError('Ollama returned an unexpected embedding shape.')\n
    return embeddings\n
\n
sample_embedding = generate_embeddings(['The School Management System uses PostgreSQL as its database.'])\n
print('Embedding generated successfully!')\n
print('Vector shape:', sample_embedding.shape)\n
print('First 10 values:', sample_embedding[0, :10])

### 4. Normalize and index vectors\n
FAISS `IndexFlatIP` performs exact inner-product search. If both the document and query vectors have L2 norm 1, inner product equals cosine similarity. Scores closer to 1 imply more similar vector direction; they are not probabilities.

In [ ]:
document_embeddings = generate_embeddings([chunk['text'] for chunk in all_chunks])\n
faiss.normalize_L2(document_embeddings)\n
\n
vector_norms = np.linalg.norm(document_embeddings, axis=1)\n
assert np.allclose(vector_norms, 1.0, atol=1e-5)\n
\n
embedding_dimension = document_embeddings.shape[1]\n
faiss_index = faiss.IndexFlatIP(embedding_dimension)\n
faiss_index.add(document_embeddings)\n
assert faiss_index.ntotal == len(all_chunks)\n
print('Embedding matrix shape:', document_embeddings.shape)\n
print('FAISS index created with vectors:', faiss_index.ntotal)\n
print('All document vectors are L2 normalized.')

### 5. Semantic search\n
The search function embeds a question with the query prefix, normalizes it, searches FAISS, and joins returned vector positions to the original chunk metadata. This is how retrieved passages retain their source and PDF page.

In [ ]:
def generate_query_embedding(query: str, model: str = EMBEDDING_MODEL) -> np.ndarray:\n
    if not isinstance(query, str) or not query.strip():\n
        raise ValueError('Query cannot be empty.')\n
    response = ollama.embed(model=model, input='search_query: ' + query.strip())\n
    vector = np.asarray(_embedding_values(response)[0], dtype=np.float32).reshape(1, -1)\n
    if vector.shape[1] != faiss_index.d:\n
        raise ValueError('Query and document embedding dimensions differ.')\n
    faiss.normalize_L2(vector)\n
    return vector\n
\n
def search_documents(query: str, top_k: int = 5) -> list[dict[str, Any]]:\n
    if top_k <= 0:\n
        raise ValueError('top_k must be positive.')\n
    scores, indices = faiss_index.search(generate_query_embedding(query), min(top_k, faiss_index.ntotal))\n
    results = []\n
    for score, index in zip(scores[0], indices[0]):\n
        if index < 0:\n
            continue\n
        chunk = all_chunks[int(index)]\n
        results.append({**chunk, 'similarity_score': float(score)})\n
    return results\n
\n
def display_search_results(results: list[dict[str, Any]]) -> pd.DataFrame:\n
    return pd.DataFrame([\n
        {'rank': rank, 'similarity_score': round(item['similarity_score'], 4), 'source': item['source'], 'page': item['page_number'], 'chunk_id': item['chunk_id'], 'text': item['text']}\n
        for rank, item in enumerate(results, start=1)\n
    ])

In [ ]:
query = 'What database does the application use?'\n
search_results = search_documents(query, top_k=3)\n
display(display_search_results(search_results))\n
\n
for rank, result in enumerate(search_results, start=1):\n
    print(f'#{rank} | {result["source"]} | page {result["page_number"]} | score {result["similarity_score"]:.4f}')\n
    print(result['text'])\n
    print('-' * 70)

### 6. Persist and verify the vector store\n
The index and metadata must be saved together in exactly the same order. On restart, the index position returned by FAISS points to the metadata record at the same list position.

In [ ]:
index_path = VECTOR_DB_DIR / 'documind.faiss'\n
metadata_path = VECTOR_DB_DIR / 'metadata.json'\n
faiss.write_index(faiss_index, str(index_path))\n
with metadata_path.open('w', encoding='utf-8') as file:\n
    json.dump(all_chunks, file, indent=2, ensure_ascii=False)\n
\n
loaded_index = faiss.read_index(str(index_path))\n
with metadata_path.open(encoding='utf-8') as file:\n
    loaded_metadata = json.load(file)\n
assert loaded_index.ntotal == len(loaded_metadata) == len(all_chunks)\n
print('Vector database saved and reloaded successfully!')\n
print('Index:', index_path)\n
print('Metadata:', metadata_path)

### 7. Baseline retrieval evaluation\n
This is a small smoke test, not a full retrieval benchmark. It checks that the expected keyword appears somewhere in the top three retrieved chunks. If it fails, inspect the retrieved passages before changing model or similarity settings.

In [ ]:
evaluation_queries = [\n
    {'question': 'Which database is used?', 'expected_keyword': 'PostgreSQL'},\n
    {'question': 'Which frontend framework is required?', 'expected_keyword': 'React'},\n
    {'question': 'Which backend framework is used?', 'expected_keyword': 'FastAPI'},\n
]\n
evaluation_results = []\n
for item in evaluation_queries:\n
    retrieved_text = ' '.join(result['text'] for result in search_documents(item['question'], top_k=3)).lower()\n
    evaluation_results.append({**item, 'passed': item['expected_keyword'].lower() in retrieved_text})\n
\n
evaluation_df = pd.DataFrame(evaluation_results)\n
display(evaluation_df)\n
print('Keyword retrieval success:', f'{evaluation_df.passed.mean():.0%}')